In [0]:
# @title library installation
!pip install openai==1.55.3

In [0]:
dbutils.library.restartPython()

In [0]:
# @title IMPORTING ALL THE NECESSARY LIBRARIES FOR PROJ
import openai
import time
import random
from openai import OpenAI
import pandas as pd
from datetime import date
from pyspark.sql.functions import current_timestamp
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import httpx
import requests
from requests.auth import HTTPBasicAuth
import json
from docx import Document
from datetime import datetime
import pytz
from collections import defaultdict
import hashlib
import threading
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

In [0]:
# Define tracker Delta table path
spark_table_path = "same_batch_track_path_mentioned_on_feedback_notebook_1"

# Read all tracked batches
status_df = spark.read.format("delta").load(spark_table_path)
status_df.show()
submitted_df = status_df.filter(status_df.status == "submitted").select("batch_id")

pending_batches = submitted_df.collect()
print(f"Pending batches to check: {len(pending_batches)}")
print(f"Pending batches to check: {pending_batches}")
if not pending_batches:
    print("No pending batches. Exiting.")
    try:
        dbutils.notebook.exit("No batches to process.")
    except:
        import sys
        sys.exit(0)

In [0]:
def exponential_backoff(retry_count, max_backoff=256):
    random_jitter = random.uniform(0, 1)
    wait_time = min((2 ** retry_count) + random_jitter, max_backoff)
    return wait_time

def update_tracker(batch_id, status, output_file_id=None):
    set_clause = f"""
        target.status = '{status}',
        target.completed_at = current_timestamp()
    """

    if status == "completed" and output_file_id:
        set_clause += f", target.output_file_id = '{output_file_id}'"

    update_sql = f"""
        MERGE INTO delta.`{spark_table_path}` AS target
        USING (SELECT '{batch_id}' AS batch_id) AS source
        ON target.batch_id = source.batch_id
        WHEN MATCHED THEN UPDATE SET
            {set_clause}
    """

    spark.sql(update_sql)
    print(f"Tracker updated: {batch_id} → {status}")

def check_batch_status(batch_id):
    retry_count = 0
    while True:
        try:
            response = requests.get(
                f"https://api.openai.com/v1/batches/{batch_id}",
                headers={"Authorization": f"Bearer {openai.api_key}"}
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            retry_count += 1
            wait_time = exponential_backoff(retry_count)
            print(f"Error checking batch status: {e}, retrying in {wait_time:.2f} seconds...")
            time.sleep(wait_time)

def retrieve_and_save_batch_results(output_file_id, output_filename='batch_output.jsonl'):
    try:
        file_response = requests.get(
            f"https://api.openai.com/v1/files/{output_file_id}/content",
            headers={"Authorization": f"Bearer {openai.api_key}"}
        )
        file_response.raise_for_status()
        with open(output_filename, 'w') as output_file:
            output_file.write(file_response.text)
        print(f"Batch results saved to '{output_filename}'")
    except requests.exceptions.RequestException as e:
        print(f"Error retrieving batch results: {e}")

In [0]:
for row in pending_batches:
    batch_id = row["batch_id"]
    print(f"\nChecking Batch ID: {batch_id}")
    
    batch_info = check_batch_status(batch_id)
    current_status = batch_info.get("status")
    print(f"Status: {current_status}")

    if current_status == "completed":
        output_file_id = batch_info.get("output_file_id")

        update_tracker(batch_id, "completed", output_file_id=output_file_id)

        retrieve_and_save_batch_results(output_file_id, output_filename='batch_output.jsonl')

        print("Batch completed and results downloaded.")

        tracker_row = status_df.filter(col("batch_id") == batch_id).collect()[0]
        all_manual_indices = json.loads(tracker_row["manual_indices_json"])
        original_manual_comments = json.loads(tracker_row["original_manual_comments_json"])
        print("Retrieved manual indices and original comments from tracker.")

    elif current_status in ["failed", "expired", "cancelled"]:
        update_tracker(batch_id, current_status)
        print(f"Batch marked as {current_status}.")

    else:
        print(f"Batch {batch_id} still processing.")
        dbutils.notebook.exit("No batches to process.")

In [0]:
#this is the dataframe that holds the comments for classification (same mentioned under 1st feedback notebook)
main_df

In [0]:
#this is the dataframe that holds the test dataset for validation [should hold 'Comment', 'Module' columns]
module_test_df = pd.read_csv('enter_the_test_dataset_path_here')

In [0]:
def clean_response(response_text):
    lines = response_text.split("\n")
    cleaned_lines = [re.sub(r"^\d+\.\s*", "", line).rstrip() for line in lines]
    return "\n".join(cleaned_lines)

def extract_classified_comments(output_filename='batch_output.jsonl'):
    classified_comments = []
    try:
        with open(output_filename, 'r') as f:
            for line in f:
                result = json.loads(line)
                classified_comment = result['response']['body']['choices'][0]['message']['content']
                cleaned_comment = clean_response(classified_comment)
                classified_comments.append(cleaned_comment)
    except Exception as e:
        print(f"Error reading batch output file: {e}")
    return classified_comments

def process_batch_output(main_df, module_test_df, all_manual_indices, original_manual_comments, output_filename='batch_output.jsonl'):
    manual_data = dict(zip(module_test_df['Comment'], module_test_df['Module']))
    classified_comments = extract_classified_comments(output_filename)

    manual_to_reclassified_dicts = []
    automated_comments_batches = []

    for batch_index, manual_indices in enumerate(all_manual_indices):
        injected_comments = original_manual_comments[batch_index]
        batch_classified_comments = classified_comments[batch_index].split("\n")

        manual_to_reclassified_dict = {}
        for i, index in enumerate(manual_indices):
            original_comment = injected_comments[i]
            reclassified_value = batch_classified_comments[index]
            manual_to_reclassified_dict[original_comment] = reclassified_value

        manual_to_reclassified_dicts.append(manual_to_reclassified_dict)

        current_automated_comments = [
            comment for i, comment in enumerate(batch_classified_comments) if i not in manual_indices
        ]
        automated_comments_batches.append(current_automated_comments)

    manual_to_reclassified_dicts_with_original = []
    for batch_dict in manual_to_reclassified_dicts:
        updated_batch_dict = {}
        for original_comment, reclassified_value in batch_dict.items():
            original_classification = manual_data.get(original_comment, "Unknown")
            updated_batch_dict[original_comment] = {
                "Original Classification": original_classification,
                "Reclassified": reclassified_value
            }
        manual_to_reclassified_dicts_with_original.append(updated_batch_dict)

    # 1. Update main_df with automated predictions
    flattened_automated_comments = [comment for batch in automated_comments_batches for comment in batch]
    updated_main_df = main_df.copy()

    if len(flattened_automated_comments) != len(updated_main_df):
        print("Prediction count mismatch. Truncating to match.")
    updated_main_df.loc[:len(flattened_automated_comments) - 1, 'module'] = flattened_automated_comments

    # 2. Build validation_df
    validation_rows = []
    for batch_number, batch_dict in enumerate(manual_to_reclassified_dicts_with_original):
        for original_comment, classifications in batch_dict.items():
            original_class = classifications["Original Classification"]
            reclassified = classifications["Reclassified"]
            is_valid = "True" if original_class in reclassified else "False"
            validation_rows.append({
                "Comment": original_comment,
                "Original Classification": original_class,
                "Reclassification": reclassified,
                "Batch Number": batch_number + 1,
                "Validation": is_valid
            })

    overall_correct = sum(1 for row in validation_rows if row['Validation'] == "True")
    overall_total = len(validation_rows)
    overall_accuracy = (overall_correct / overall_total) * 100 if overall_total > 0 else 0

    validation_rows.append({
        "Comment": "Overall Accuracy",
        "Original Classification": "",
        "Reclassification": "",
        "Batch Number": "All Batches",
        "Validation": f"{overall_accuracy:.2f}%"
    })

    validation_df = pd.DataFrame(validation_rows)
    return updated_main_df, validation_df

In [0]:
retrieve_and_save_batch_results(output_file_id, output_filename='batch_output.jsonl')

main_df_updated, validation_df = process_batch_output(
    main_df=main_df,
    module_test_df=module_test_df,
    all_manual_indices=all_manual_indices,
    original_manual_comments=original_manual_comments,
    output_filename='batch_output.jsonl'
)

In [0]:
#this is the dataframe that holds the raw comments for classification (same mentioned under 1st feedback notebook)
main_df_updated

In [0]:
validation_df.display()

In [0]:
# Step 1: Clean column names
main_df_updated.columns = [
    col.strip()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "")
        .replace(";", "")
        .replace("{", "")
        .replace("}", "")
        .replace("=", "")
        .replace("\n", "")
        .replace("\t", "")
    for col in main_df_updated.columns
]

In [0]:
# Get today's date for classification run
classification_date = date.today()

# Get review date from main_df_updated (assuming uniform 'dt1')
review_date = pd.to_datetime(main_df_updated['dt1'].iloc[0]).date()

# Add to validation_df
validation_df["classification_date"] = classification_date
validation_df["review_date"] = review_date

validation_df = validation_df.rename(columns={
    "Comment": "comment",
    "Original Classification": "original_classification",
    "Reclassification": "reclassification",
    "Batch Number": "batch_number",
    "Validation": "validation"
})

In [0]:
validation_spark_df = spark.createDataFrame(validation_df)
validation_spark_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("accuracy_table_path")

In [0]:
# prompt 
system_instructions = """
MENTION_THE_EXACT_PROMPT_MENTIONED_UNDER_NOTEBOOK_1
"""

In [0]:
# to handle misclassification of modules and reclassifying them and multiple modules for a single comment
DEFINED_MODULES = [
    "Home",
    "Search",
    "Product",
    "Cart",
    "Checkout",
    "Payment",
    "Order"
] #this is the list of modules that are defined in the prompt just for a sample (can mention as per the usecase)

# Function to generate response using OpenAI's chat completions API
def generate_response(system_instructions, comments):
    full_system_instructions = system_instructions + "\n Please provide the output exactly like this - index: Module"
    messages = [
        {"role": "system", "content": full_system_instructions},
        {"role": "user", "content": comments}
    ]
    try:
        completion = openai.chat.completions.create(
            model="gpt-4.1-2025-04-14",
            messages=messages,
            temperature=0.2
        )
        response_text = completion.choices[0].message.content

        print("Generated Response:\n", response_text)
        return response_text

    except Exception as e:
        print(f"An error occurred in generate_response: {e}")
        return None

# Step 1: Remove "General Feedback" as a secondary classification
def remove_general_feedback(df):
    def clean_module(module):
        modules = [m.strip() for m in module.split(",")]
        if "Generic Feedback" in modules and len(modules) > 1:
            modules.remove("Generic Feedback")
        return ", ".join(modules)

    df["module"] = df["module"].apply(clean_module)
    return df

# Step 2: Identify and extract out-of-scope modules
def extract_out_of_scope(df):
    out_of_scope = df[~df["module"].apply(lambda x: all(m.strip() in DEFINED_MODULES for m in x.split(",")))]
    print("out of scope:\n", out_of_scope)
    in_scope = df[df.index.isin(out_of_scope.index) == False]
    print("in scope:\n", in_scope)
    return in_scope, out_of_scope

# Step 3: Reclassify out-of-scope modules using OpenAI's API
def reclassify_out_of_scope(out_of_scope):
    if out_of_scope.empty:
        return out_of_scope  # No out-of-scope rows to reclassify

    # Prepare comments for classification
    comments_data = "\n".join([f"{idx}: {row['comment']}" for idx, row in out_of_scope.iterrows()])
    # Use the provided generate_response function
    response = generate_response(system_instructions, comments_data)

    if response:
        # Parse GPT output
        reclassified = {}
        for line in response.strip().split("\n"):
            parts = re.split(r"\s*:\s*", line, maxsplit=1)
            if len(parts) != 2:
                print(f"Skipping malformed line: {line}")
                continue
            idx, module = parts
            reclassified[int(idx)] = module.strip()

        # Update DataFrame with reclassified modules
        for idx, module in reclassified.items():
            out_of_scope.loc[idx, "module"] = module

    return out_of_scope

# Step 4: Combine updated rows with the original DataFrame
def update_dataframe(in_scope, reclassified_out_of_scope):
    return pd.concat([in_scope, reclassified_out_of_scope]).sort_index()

# Process
main_df_updated = remove_general_feedback(main_df_updated)
in_scope, out_of_scope = extract_out_of_scope(main_df_updated)
reclassified_out_of_scope = reclassify_out_of_scope(out_of_scope)
main_df_updated = update_dataframe(in_scope, reclassified_out_of_scope)

# Display Final DataFrame
print("Final DataFrame:\n", main_df_updated)

# @title CLEANING THE MODULE AND DUPLICATING, APPENDING and INDEXING the 2nd and 3rd modules (Multiple modules)
def enforce_text_format(value):
    # If value is a very long numeric-like string, treat it as text
    if isinstance(value, int) or (isinstance(value, str) and value.isdigit() and len(value) > 15):
        return f"'{value}"  # Prefix with single quote to enforce text format in Sheets
    return value

def expand_categories(df):
    expanded_rows = []
    for idx, row in df.iterrows():
        modules = re.split(r",\s*", row['module'])
        for i, module in enumerate(modules):
            new_row = row.copy()
            new_row['module'] = module
            if len(modules) == 1:  # Only one category
                new_row['module_classification'] = "1"
            elif len(modules) == 2:  # Two categories
                new_row['module_classification'] = "1,2" if i == 0 else "2"
            elif len(modules) == 3:  # Three categories
                new_row['module_classification'] = "1,2,3" if i == 0 else ("2,3" if i == 1 else "3")
            expanded_rows.append(new_row)

    return pd.DataFrame(expanded_rows)

# Custom sort key function for Module Classification
def sorting_key(classification):
    if classification == "1,2,3":
        return (1, 2, 3)
    elif classification == "1,2":
        return (1, 2)
    elif classification == "1":
        return (1,)
    elif classification == "2,3":
        return (2, 3)
    elif classification == "2":
        return (2,)
    else:
        return (3,)

expanded_df = expand_categories(main_df_updated)
expanded_df['Sorting Key'] = expanded_df['module_classification'].apply(sorting_key)
expanded_df = expanded_df.sort_values(by='Sorting Key')
expanded_df = expanded_df.drop(columns=['Sorting Key'])

# Apply enforce_text_format only to the 'Comment' column
expanded_df['comment'] = expanded_df['comment'].apply(enforce_text_format)
expanded_df = expanded_df.replace([np.inf, -np.inf], np.finfo(np.float64).max)

In [0]:
# classified comments where the classification is metioned under 'module' column
expanded_df

In [0]:
# Step 2: Convert to Spark DataFrame
expanded_spark_df = spark.createDataFrame(expanded_df)

# Step 3: Write to Delta table in Unity Catalog
expanded_spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("enter_the_final_analyzed_table_path_here")